# TKG Solar Forecasting — Results Analysis
Pred-vs-actual and per-horizon error for a trained model.

Run from the project root after training (e.g. `python main.py --config configs/smoke_config.yaml`).
See `docs/assumptions.md` for the data-mismatch caveat (absolute reproduction not expected).

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd()))
import torch, numpy as np, matplotlib.pyplot as plt
from src.common.seeding import seed_everything
from src.common.shapes import HORIZON_MINUTES
from src.data_pipeline import DataPipeline
from src.training.config import Config
from src.training.train_loop import predict_loader
from src.fusion_predictor.tkg_solar_model import TKGSolarModel
seed_everything(42)

In [ ]:
cfg = Config.from_yaml('configs/smoke_config.yaml')
splits = DataPipeline.load(cfg.opsd_path, cfg.nsrdb_path, cfg.himawari_dir,
                         k=cfg.k, batch_size=cfg.batch_size, img_size=cfg.img_size, min_steps=cfg.min_steps)
model = TKGSolarModel.from_config(cfg)
ckpt = pathlib.Path(cfg.checkpoint_dir) / 'best_full.pt'
if ckpt.exists():
    model.load_state_dict(torch.load(ckpt, map_location='cpu')['model_state'])
yt, yp = predict_loader(model, splits.test_loader, 'cpu')
yt = splits.scalers.inverse_pv(yt.numpy()); yp = splits.scalers.inverse_pv(yp.numpy())

In [ ]:
fig, axes = plt.subplots(1, len(HORIZON_MINUTES), figsize=(15, 4))
for j, mins in enumerate(HORIZON_MINUTES):
    axes[j].plot(yt[:150, j], label='actual')
    axes[j].plot(yp[:150, j], label='pred', alpha=0.7)
    axes[j].set_title(f'{mins}-min horizon'); axes[j].legend()
plt.tight_layout(); plt.show()

In [ ]:
from src.metrics.regression_metrics import compute_per_horizon
m = compute_per_horizon(yt, yp, HORIZON_MINUTES, cfg.mape_min_value)
for k, v in m.items():
    print(k, {kk: round(vv, 4) for kk, vv in v.items()})